# Focus Alternative Results Verification

## Unit of analysis
- Each observation is a context-trigger-query tuple. 
- Within a context, an observation is a trigger-query pair. 
- Each observation has a model predicited exclusion probability and a human exclusion rate. 
- 16 contexts, 30 trigger-query pairs per context. 480 total context-trigger-query tuples. 

## Calculations to verify
- Pearson correlation within each context. 
- Pearson correlation across all 480 tuples pooled. 
- Spearman correlation within each context, and its mean across contexts.
- Pair-level log scores and their averages.

## Load and inspect data 

In [21]:
from pathlib import Path
import pandas as pd

repo_root = Path("/Users/lailajohnston/Documents/Research Projects /FocusAlternatives/structure-of-alternatives")

results_file = (
    repo_root
    / "focus_alt_exp_pipeline/results/set_variant_qwen_corrected_2026-09-15"
    / "advisor_summary/item_level_scatterplots/novel_focus_by_context"
    / "novel_focus_item_level_predictions.csv"
)

results = pd.read_csv(results_file)

print("Rows and columns:", results.shape)
results.head()

# 9 models x 480 total context-trigger-query tuples = 4,320 rows

Rows and columns: (4320, 13)


,model,analysis_dataset_id,analysis_label,analysis_unit_id,human_rate,prediction,item_log_score,exponentiated_log_score,dataset,item_id,context_id,trigger,query
0,No linking structure,novel_focus,Novel focus alternatives,bag::cash::chalk,1.000000,0.000027,-10.517298,0.000027,Novel Focus Alternative Study,bag::cash::chalk,bag,cash,chalk
1,No linking structure,novel_focus,Novel focus alternatives,bag::cash::crayons,1.000000,0.000187,-8.586970,0.000187,Novel Focus Alternative Study,bag::cash::crayons,bag,cash,crayons
2,No linking structure,novel_focus,Novel focus alternatives,bag::cash::headphones,0.000000,0.000002,-0.000002,0.999998,Novel Focus Alternative Study,bag::cash::headphones,bag,cash,headphones
3,No linking structure,novel_focus,Novel focus alternatives,bag::cash::makeup,0.105263,0.000019,-1.146574,0.317723,Novel Focus Alternative Study,bag::cash::makeup,bag,cash,makeup
4,No linking structure,novel_focus,Novel focus alternatives,bag::cash::pen,1.000000,0.019163,-3.954797,0.019163,Novel Focus Alternative Study,bag::cash::pen,bag,cash,pen


In [22]:
results.groupby("model").size()

model
Conjunction Top-K       480
Conjunction Top-p       480
Disjunction Top-K       480
Disjunction Top-p       480
No linking structure    480
Ordering                480
Set Top-K               480
Set Top-p               480
X but not Y             480
dtype: int64

# No Linking Model Pearson Correlation Inspections

## Pooled Pearson Calculation: No Linking

Pearson correlation between the No linking model’s predictions and human exclusion rates across all 480 pairs, pooled across the 16 contexts. Each pair has equal weight.

In [23]:
import sys
import math

code_directory = repo_root / "focus_alt_exp_pipeline" / "code"
sys.path.insert(0, str(code_directory))

from build_results_viewer import mean, pearson

# Mean function that is used in the results viewer

# def mean(values):
#     values = list(values)
#     return math.fsum(values) / len(values) if values else None


# Pearson function that is used in the results viewer

# def pearson(xs, ys):
#     if len(xs) < 2:
#         return None
#     mx, my = mean(xs), mean(ys)
#     dx, dy = [x - mx for x in xs], [y - my for y in ys]
#     denominator = math.sqrt(math.fsum(x*x for x in dx) * math.fsum(y*y for y in dy))
#     if not denominator:
#         return None
#     return max(-1., min(1., math.fsum(x*y for x, y in zip(d
# x, dy)) / denominator))

In [24]:
# Inspect the No linking structure data 

no_linking = results.loc[
    results["model"] == "No linking structure"
].copy()

print("Number of pairs:", len(no_linking))
print("Number of contexts:", no_linking["context_id"].nunique())

xs = no_linking["prediction"].tolist()
ys = no_linking["human_rate"].tolist()

print("Number of model values:", len(xs))
print("Number of human values:", len(ys))
print("Fewer than two observations?", len(xs) < 2)

# How to account for this difference in value ranges?

ranges = pd.DataFrame({
    "minimum": [min(xs), min(ys)],
    "maximum": [max(xs), max(ys)]
}, index=["Model predictions", "Human rates"])

ranges["range_width"] = ranges["maximum"] - ranges["minimum"]

ranges

Number of pairs: 480
Number of contexts: 16
Number of model values: 480
Number of human values: 480
Fewer than two observations? False


,minimum,maximum,range_width
Model predictions,1.058005e-07,0.294982,0.294982
Human rates,0.000000e+00,1.000000,1.000000


In [25]:
# Inspect means 

mx, my = mean(xs), mean(ys)

print("Mean model probability:", mx)
print("Mean human rate:", my)


# Substract each mean from the corresponding values to get deviations from the mean. 

# dx: how far each model prediction is above or below the model mean.
# dy: how far each human rate is above or below the human mean.

dx, dy = [x - mx for x in xs], [y - my for y in ys]

pd.DataFrame({
    "model_probability": xs,
    "model_deviation": dx,
    "human_rate": ys,
    "human_deviation": dy
}).head(10)

# model deviation = model probability - mean model probability
# human deviation = human rate - mean human rate

Mean model probability: 0.018164526545278316
Mean human rate: 0.5040436414183511


,model_probability,model_deviation,human_rate,human_deviation
0,0.000027,-0.018137,1.000000,0.495956
1,0.000187,-0.017978,1.000000,0.495956
2,0.000002,-0.018162,0.000000,-0.504044
3,0.000019,-0.018146,0.105263,-0.398780
4,0.019163,0.000998,1.000000,0.495956
5,0.000001,-0.018163,0.000000,-0.504044
6,0.000187,-0.017978,1.000000,0.495956
7,0.000002,-0.018162,0.200000,-0.304044
8,0.000019,-0.018146,0.200000,-0.304044
9,0.019163,0.000998,1.000000,0.495956


In [26]:
# Calculate the denominator for the Pearson correlation coefficient formula.
# The denominator is the product of the square roots of the sums of squares of the deviations.

denominator = math.sqrt(
    math.fsum(x*x for x in dx) 
    * math.fsum(y*y for y in dy)
)

print("Sum of squared model deviations:", math.fsum(x*x for x in dx))
print("Sum of squared human deviations:", math.fsum(y*y for y in dy))
print("Denominator:", denominator)
print("Is the denominator zero?", not denominator)


Sum of squared model deviations: 1.3519647838301938
Sum of squared human deviations: 67.48896823228344
Denominator: 9.552104917089316
Is the denominator zero? False


In [27]:
# Calculate numerator and compute Pearson correlation. 

numerator = math.fsum(x*y for x, y in zip(dx, dy))
raw_correlation = numerator / denominator
walkthrough_result = max(-1., min(1., raw_correlation))

print("Sum of products of deviations:", numerator)

function_result = pearson(xs, ys)

print("Walkthrough, Final Pearson correlation:", walkthrough_result)
print("Imported function, Final Pearson correlation:", function_result)
print("Exactly equal?", walkthrough_result == function_result)

Sum of products of deviations: 3.4479354731026137
Walkthrough, Final Pearson correlation: 0.3609608042447315
Imported function, Final Pearson correlation: 0.3609608042447315
Exactly equal? True


## Within-Context Pearson Correlation: No Linking

Within-context Pearson centers and scales the values separately in each context. Pooled Pearson centers and scales all 480 values together.

In [33]:
context_results = []

for context, group in no_linking.groupby("context_id"):
    model_values = group["prediction"].tolist()
    human_values = group["human_rate"].tolist()

    context_results.append({
        "context": context,
        "n": len(group),
        "pearson": pearson(model_values, human_values),
        "model_mean": mean(model_values),
        "human_mean": mean(human_values),
        "model_sd": group["prediction"].std(ddof=0),
        "human_sd": group["human_rate"].std(ddof=0),
        "model_min": min(model_values),
        "model_max": max(model_values),
        "model_range_width": max(model_values) - min(model_values),
        "human_min": min(human_values),
        "human_max": max(human_values),
        "human_range_width": max(human_values) - min(human_values)
    })

context_results = pd.DataFrame(context_results).set_index("context")
context_results

,n,pearson,model_mean,human_mean,model_sd,human_sd,model_min,model_max,model_range_width,human_min,human_max,human_range_width
context,,,,,,,,,,,,
bag,30,0.499928,0.003233,0.516790,0.007124,0.437314,1.226866e-06,0.019163,0.019161,0.000000,1.0,1.000000
bakery,30,0.753645,0.008793,0.565098,0.011159,0.350949,6.883000e-05,0.026914,0.026845,0.000000,1.0,1.000000
beach,30,0.668500,0.013885,0.556515,0.019106,0.323658,1.918589e-04,0.051560,0.051369,0.000000,1.0,1.000000
cold,30,0.678622,0.029777,0.347827,0.066008,0.337221,4.884263e-06,0.177375,0.177370,0.000000,1.0,1.000000
cut,30,0.696378,0.048142,0.406397,0.104601,0.333412,1.584658e-06,0.282013,0.282011,0.000000,1.0,1.000000
fridge,30,0.557330,0.011394,0.544722,0.020212,0.428587,1.179044e-06,0.055845,0.055843,0.000000,1.0,1.000000
gym,30,0.347251,0.026089,0.721228,0.050101,0.319786,3.487622e-05,0.137731,0.137696,0.000000,1.0,1.000000
hot,30,0.174590,0.001564,0.271464,0.002147,0.327267,9.452665e-06,0.004600,0.004590,0.000000,1.0,1.000000
mall,30,0.517239,0.000260,0.596974,0.000259,0.345008,1.058005e-07,0.000577,0.000576,0.000000,1.0,1.000000


In [31]:
pooled_pearson = pearson(
    no_linking["prediction"].tolist(),
    no_linking["human_rate"].tolist()
)

print("Pooled Pearson:", pooled_pearson)

print(
    "Mean within-context Pearson:",
    mean(context_results["pearson"].tolist())
)

Pooled Pearson: 0.3609608042447315
Mean within-context Pearson: 0.5417050505757476


## Resons for differences in pooled and within-context pearson 

Pooled pearson uses one mean and one scale for the entire correlation calculation. Within-context pearson uses 16 different means and scales. 

### Differences in standard deviation 

No linking’s predictions are tightly clustered in some contexts and much more spread out in others (see model_sd above). No linking's spread in science is about 1,091 times its spread in mask. Within-context Pearson uses each context’s own denominator. For mask, it normalizes using mask’s very small spread. For science, it normalizes using science’s much larger spread.

### Inspecting context means

In the Pearson equation, we first need to calculate the deivations from the mean. For within-context, each deivation value is caluclated from that contexts mean (30 observations), and each context has a different mean. For pooled, the deivation is calculated from the mean of all 480 observations. 

mx, my = mean(xs), mean(ys)

dx, dy = [x - mx for x in xs], [y - my for y in ys]


In [ ]:
# Example context using No linking model

bag = no_linking.loc[no_linking["context_id"] == "bag"]
bag_model_mean = mean(bag["prediction"].tolist())

pooled_model_mean = mean(no_linking["prediction"].tolist())

print("Mean model prediction in bag:", bag_model_mean)
print("Mean model prediction across all contexts:", pooled_model_mean)

# Here we see that becuase the means are different, the deviations from the mean will be different.
# Pooled mean is higher than the mean for the bag context.

Mean model prediction in bag: 0.0032330311600904152
Mean model prediction across all contexts: 0.018164526545278316


In [ ]:
# Inspect one observation from the bag context to see how the deviations from the mean are calculated.

row = bag.iloc[0]
x = row["prediction"]

print("Pair:", row["trigger"], "→", row["query"])
print("No Linking Prediction:", x)
print("Deviation from bag No linking mean:", x - bag_model_mean)
print("Deviation from pooled No linking mean:", x - pooled_model_mean)

# Same prediction value (cash -> chalk) is farther below the pooled mean than the bag mean.

Pair: cash → chalk
No Linking Prediction: 2.706422684422374e-05
Deviation from bag No linking mean: -0.0032059669332461915
Deviation from pooled No linking mean: -0.01813746231843409


In [ ]:
# Human rates in bad context.

y = row["human_rate"]

bag_human_mean = mean(bag["human_rate"].tolist())
pooled_human_mean = mean(no_linking["human_rate"].tolist())

print("Human rate for cash → chalk:", y)

print("Mean human rate in bag:", bag_human_mean)
print("Mean human rate across all contexts:", pooled_human_mean)

print("Deviation from bag human mean:", y - bag_human_mean)
print("Deviation from pooled human mean:", y - pooled_human_mean)

# Human mean within the bag context is similar to the pooled human mean, so the deviations from the mean are similar.

Human rate for cash → chalk: 1.0
Mean human rate in bag: 0.5167899323472079
Mean human rate across all contexts: 0.5040436414183511
Deviation from bag human mean: 0.48321006765279206
Deviation from pooled human mean: 0.4959563585816489


In [44]:
deviation_comparison = pd.DataFrame({
    "model_deviation": [
        x - bag_model_mean,
        x - pooled_model_mean
    ],
    "human_deviation": [
        y - bag_human_mean,
        y - pooled_human_mean
    ]
}, index=["Within bag", "Pooled"])

print("Deviations from the mean for cash → chalk in bag context.")
print("The product of the deviations is used to calculate the numerator for the Pearson correlation.")

deviation_comparison["product"] = (
    deviation_comparison["model_deviation"]
    * deviation_comparison["human_deviation"]
)

deviation_comparison

Deviations from the mean for cash → chalk in bag context.
The product of the deviations is used to calculate the numerator for the Pearson correlation.


,model_deviation,human_deviation,product
Within bag,-0.003206,0.483210,-0.001549
Pooled,-0.018137,0.495956,-0.008995


In [ ]:
# Repeat calculation for all 30 observations in the bag context.

bag_products = bag[
    ["trigger", "query", "prediction", "human_rate"]
].copy()

bag_products["bag_model_deviation"] = (
    bag_products["prediction"] - bag_model_mean
)

bag_products["bag_human_deviation"] = (
    bag_products["human_rate"] - bag_human_mean
)

bag_products["within_bag_product"] = (
    bag_products["bag_model_deviation"]
    * bag_products["bag_human_deviation"]
)

bag_products["pooled_model_deviation"] = (
    bag_products["prediction"] - pooled_model_mean
)

bag_products["pooled_human_deviation"] = (
    bag_products["human_rate"] - pooled_human_mean
)

bag_products["pooled_product"] = (
    bag_products["pooled_model_deviation"]
    * bag_products["pooled_human_deviation"]
)

bag_products

,trigger,query,prediction,human_rate,bag_model_deviation,bag_human_deviation,within_bag_product,pooled_model_deviation,pooled_human_deviation,pooled_product
0,cash,chalk,0.000027,1.000000,-0.003206,0.483210,-0.001549,-0.018137,0.495956,-0.008995
1,cash,crayons,0.000187,1.000000,-0.003047,0.483210,-0.001472,-0.017978,0.495956,-0.008916
2,cash,headphones,0.000002,0.000000,-0.003231,-0.516790,0.001670,-0.018162,-0.504044,0.009155
3,cash,makeup,0.000019,0.105263,-0.003214,-0.411527,0.001323,-0.018146,-0.398780,0.007236
4,cash,pen,0.019163,1.000000,0.015930,0.483210,0.007697,0.000998,0.495956,0.000495
5,chalk,cash,0.000001,0.000000,-0.003232,-0.516790,0.001670,-0.018163,-0.504044,0.009155
6,chalk,crayons,0.000187,1.000000,-0.003047,0.483210,-0.001472,-0.017978,0.495956,-0.008916
7,chalk,headphones,0.000002,0.200000,-0.003231,-0.316790,0.001023,-0.018162,-0.304044,0.005522
8,chalk,makeup,0.000019,0.200000,-0.003214,-0.316790,0.001018,-0.018146,-0.304044,0.005517
9,chalk,pen,0.019163,1.000000,0.015930,0.483210,0.007697,0.000998,0.495956,0.000495


In [ ]:
bag_numerator = math.fsum(bag_products["within_bag_product"])
bag_contribution_to_pooled = math.fsum(bag_products["pooled_product"])

print("Sum the 30 products from within the bag context to get the within-bag numerator.")
print("Within-bag numerator:", bag_numerator)
print()

print("Sum the 30 products from the pooled means to get bag's contribution to the pooled numerator.")
print("Pooled numerator also includes the products from the other 15 contexts.")
print("Bag's contribution to the pooled numerator:", bag_contribution_to_pooled)


Sum the 30 products from within the bag context to get the within-bag numerator.
Within-bag numerator: 0.0467258083596334

Sum the 30 products from the pooled means to get bag's contribution to the pooled numerator.
Pooled numerator also includes the products from the other 15 contexts.
Bag's contribution to the pooled numerator: 0.04101617283415879


In [ ]:
# Calculate within-bag denominator. 

bag_model_squared_sum = math.fsum(
    d*d for d in bag_products["bag_model_deviation"]
)

bag_human_squared_sum = math.fsum(
    d*d for d in bag_products["bag_human_deviation"]
)

bag_denominator = math.sqrt(
    bag_model_squared_sum * bag_human_squared_sum
)

print("Sum of squared model deviations:", bag_model_squared_sum)
print("Sum of squared human deviations:", bag_human_squared_sum)
print("Within-bag denominator:", bag_denominator)

Sum of squared model deviations: 0.0015226218505066979
Sum of squared human deviations: 5.737296624973906
Within-bag denominator: 0.09346514432676815


In [52]:
# Divide the numerator by the denominator.

bag_pearson_manual = bag_numerator / bag_denominator

print("Within-bag Pearson:", bag_pearson_manual)

bag_pearson_function = pearson(
    bag["prediction"].tolist(),
    bag["human_rate"].tolist()
)

print("Already existing function:", bag_pearson_function)

print(
    "Results agree:",
    math.isclose(
        bag_pearson_manual,
        bag_pearson_function,
        rel_tol=1e-12
    )
)

Within-bag Pearson: 0.499927632875342
Already existing function: 0.499927632875342
Results agree: True


## Part way conclusions: No linking, Pearson 

1. Reproduced the Pearson arithmetic. 
* The step-by-step calculations I did here match the exsiting function / values I have in my results. 

2. No linking substantially underpredicts human exclusion on average (we already knew this).
* Pooled mean no linking exclusion probability: 0.0182
* Pooled mean human exclusion rate: 0.5040

3. Contexts have differences in means and spread (for the model predicitions!)
* Human predicitions across contexts are much less variable than the model predicitions. Human predicitions are in the range of 0-1, and across contexts the standard deivations are closer than in the model standard deivations. 
* Context means differ. Model-context standard deviations differ a lot! 
* Pooled and within-context Pearson use different reference means and normalization terms. 

I haven't isolated the effects of centering versus scaling or repeated the comparison for other models.

### Centering 

Subtract each context’s own model mean and human mean from its observations, then pool all 480 centered observations.

In [54]:
columns = ["prediction", "human_rate"]

context_means = (
    no_linking.groupby("context_id")[columns]
    .transform("mean")
)

centered_values = no_linking[columns] - context_means

centered_values

,prediction,human_rate
0,-0.003206,0.483210
1,-0.003047,0.483210
2,-0.003231,-0.516790
3,-0.003214,-0.411527
4,0.015930,0.483210
...,...,...
475,0.095815,0.540483
476,-0.028414,0.040483
477,-0.028400,-0.259517
478,-0.028107,-0.459517


In [55]:
centered_pearson = pearson(
    centered_values["prediction"].tolist(),
    centered_values["human_rate"].tolist()
)

print("Original pooled Pearson:", pooled_pearson)
print("Context-centered pooled Pearson:", centered_pearson)

Original pooled Pearson: 0.3609608042447315
Context-centered pooled Pearson: 0.38336394268646146


Removing differences between context means raises Pearson, from 0.361 to 0.383.

Context mean differences affect the pooled correlation. Centering changes both the numerator and denominator.

The centered correlation remains below the mean within-context correlation of 0.542. The contexts still have different spreads.

### Standardizing (Z-Score)

Divide each centered value by its context’s standard deviation.

In [57]:
columns = ["prediction", "human_rate"]

context_sds = (
    no_linking.groupby("context_id")[columns]
    .transform(lambda values: values.std(ddof=0))
)

standardized_values = centered_values / context_sds

print("Each context`s model and human values has a mean of approximately zero and a standard deviation of one")
standardized_values

Each context`s model and human values has a mean of approximately zero and a standard deviation of one


,prediction,human_rate
0,-0.450012,1.104951
1,-0.427629,1.104951
2,-0.453499,-1.181738
3,-0.451199,-0.941033
4,2.235977,1.104951
...,...,...
475,2.084459,1.508526
476,-0.618146,0.112990
477,-0.617854,-0.724332
478,-0.611478,-1.282546


In [58]:
standardized_pearson = pearson(
    standardized_values["prediction"].tolist(),
    standardized_values["human_rate"].tolist()
)

mean_within_pearson = mean(context_results["pearson"].tolist())

print("Original pooled Pearson:", pooled_pearson)
print("Context-centered pooled Pearson:", centered_pearson)
print("Context-standardized pooled Pearson:", standardized_pearson)
print("Mean within-context Pearson:", mean_within_pearson)

Original pooled Pearson: 0.3609608042447315
Context-centered pooled Pearson: 0.38336394268646146
Context-standardized pooled Pearson: 0.5417050505757475
Mean within-context Pearson: 0.5417050505757476


## Conclusions, No linking, Pearson 

1. Context-contered pooled Pearson shows that differences between context means have a modest net effect.
    * Removing context mean differences raises the correlation by approximately 0.022. Contribute to the gap, but do not account for the full difference.

2. Centering and normalizing within contexts makes a larger difference.
    * After centering, standardizing the contexts raises the correlation from 0.383 to 0.542. This demonstrates that the contexts’ differing spreads affect how the Pearson correlation.

3. In general, Pooled Pearson and mean within-context Pearson describe different aspects of the data.
    * Pooled Pearson: assesses association across all pairs using their original probability values.
    * Mean within-context Pearson: assesses association separately within each context, then gives every context equal weight.


# Set TopK Pearson Correlation 

## Pooled Pearson

In [ ]:
set_topk = results.loc[
    results["model"] == "Set Top-K"
].copy()

set_topk_pooled_model_mean = mean(set_topk["prediction"].tolist())
set_topk_pooled_human_mean = mean(set_topk["human_rate"].tolist())

set_topk_pooled_terms = set_topk[
    ["context_id", "trigger", "query", "prediction", "human_rate"]
].copy()

set_topk_pooled_terms["model_deviation"] = (
    set_topk_pooled_terms["prediction"] - set_topk_pooled_model_mean
)

set_topk_pooled_terms["human_deviation"] = (
    set_topk_pooled_terms["human_rate"] - set_topk_pooled_human_mean
)

set_topk_pooled_terms["product"] = (
    set_topk_pooled_terms["model_deviation"]
    * set_topk_pooled_terms["human_deviation"]
)

print("Set Top-K pooled model mean:", set_topk_pooled_model_mean)
print("Pooled human mean:", set_topk_pooled_human_mean)

set_topk_pooled_terms

Set Top-K pooled model mean: 0.18395833333333333
Set Top-K pooled human mean: 0.5040436414183511


,context_id,trigger,query,prediction,human_rate,model_deviation,human_deviation,product
960,bag,cash,chalk,0.014,1.000000,-0.169958,0.495956,-0.084292
961,bag,cash,crayons,0.068,1.000000,-0.115958,0.495956,-0.057510
962,bag,cash,headphones,0.000,0.000000,-0.183958,-0.504044,0.092723
963,bag,cash,makeup,0.008,0.105263,-0.175958,-0.398780,0.070169
964,bag,cash,pen,0.990,1.000000,0.806042,0.495956,0.399761
...,...,...,...,...,...,...,...,...
1435,transport,roller blades,car,0.986,1.000000,0.802042,0.495956,0.397778
1436,transport,roller blades,chairs,0.000,0.500000,-0.183958,-0.004044,0.000744
1437,transport,roller blades,hammer,0.000,0.200000,-0.183958,-0.304044,0.055931
1438,transport,roller blades,lawnmower,0.020,0.000000,-0.163958,-0.504044,0.082642


In [61]:
set_topk_pooled_numerator = math.fsum(
    set_topk_pooled_terms["product"]
)

pd.DataFrame({
    "number_of_pairs": [len(set_topk_pooled_terms)],
    "sum_of_products": [set_topk_pooled_numerator]
}, index=["Set Top-K pooled numerator"])

,number_of_pairs,sum_of_products
Set Top-K pooled numerator,480,31.016515


In [62]:
set_topk_pooled_terms["model_squared_deviation"] = (
    set_topk_pooled_terms["model_deviation"] ** 2
)

set_topk_pooled_terms["human_squared_deviation"] = (
    set_topk_pooled_terms["human_deviation"] ** 2
)

set_topk_pooled_terms[
    [
        "context_id", "trigger", "query",
        "model_deviation", "model_squared_deviation",
        "human_deviation", "human_squared_deviation"
    ]
]

,context_id,trigger,query,model_deviation,model_squared_deviation,human_deviation,human_squared_deviation
960,bag,cash,chalk,-0.169958,0.028886,0.495956,0.245973
961,bag,cash,crayons,-0.115958,0.013446,0.495956,0.245973
962,bag,cash,headphones,-0.183958,0.033841,-0.504044,0.254060
963,bag,cash,makeup,-0.175958,0.030961,-0.398780,0.159026
964,bag,cash,pen,0.806042,0.649703,0.495956,0.245973
...,...,...,...,...,...,...,...
1435,transport,roller blades,car,0.802042,0.643271,0.495956,0.245973
1436,transport,roller blades,chairs,-0.183958,0.033841,-0.004044,0.000016
1437,transport,roller blades,hammer,-0.183958,0.033841,-0.304044,0.092443
1438,transport,roller blades,lawnmower,-0.163958,0.026882,-0.504044,0.254060


In [63]:
set_topk_model_squared_sum = math.fsum(
    set_topk_pooled_terms["model_squared_deviation"]
)

set_topk_human_squared_sum = math.fsum(
    set_topk_pooled_terms["human_squared_deviation"]
)

set_topk_pooled_denominator = math.sqrt(
    set_topk_model_squared_sum * set_topk_human_squared_sum
)

pd.DataFrame({
    "sum_squared_model_deviations": [set_topk_model_squared_sum],
    "sum_squared_human_deviations": [set_topk_human_squared_sum],
    "denominator": [set_topk_pooled_denominator]
}, index=["Set Top-K pooled"])

,sum_squared_model_deviations,sum_squared_human_deviations,denominator
Set Top-K pooled,48.917239,67.488968,57.457584


In [67]:
set_topk_pooled_manual = (
    set_topk_pooled_numerator / set_topk_pooled_denominator
)

print(pd.DataFrame({
    "numerator": [set_topk_pooled_numerator],
    "denominator": [set_topk_pooled_denominator],
    "pearson": [set_topk_pooled_manual]
}, index=["Set Top-K pooled"]))

set_topk_pooled_function = pearson(
    set_topk["prediction"].tolist(),
    set_topk["human_rate"].tolist()
)

print()
print(
    "Results agree:",
    math.isclose(
        set_topk_pooled_manual,
        set_topk_pooled_function,
        rel_tol=1e-12
    )
)

                  numerator  denominator   pearson
Set Top-K pooled  31.016515    57.457584  0.539816

Results agree: True


### Conclusions 

Across the same 480 observations, Set Top-K has a higher pooled Pearson correlation with human exclusion rates (0.540) than No linking (0.361). This indicates stronger pooled linear association. 

Set Top-K has a larger numerator and denominator than No linking, but their ratio is higher. Its higher correlation is not simply a consequence of having larger probabilities - multiplying every prediction by a positive constant would leave Pearson unchanged.

## Bag Context Pearson

In [ ]:
set_topk_bag = set_topk.loc[
    set_topk["context_id"] == "bag"
].copy()

set_topk_bag_model_mean = mean(set_topk_bag["prediction"].tolist())
set_topk_bag_human_mean = mean(set_topk_bag["human_rate"].tolist())

set_topk_pooled_model_mean = mean(set_topk["prediction"].tolist())
set_topk_pooled_human_mean = mean(set_topk["human_rate"].tolist())

pd.DataFrame({
    "number_of_pairs": [len(set_topk_bag), len(set_topk)],
    "model_mean": [
        set_topk_bag_model_mean,
        set_topk_pooled_model_mean
    ],
    "human_mean": [
        set_topk_bag_human_mean,
        set_topk_pooled_human_mean
    ]
}, index=["Bag", "All contexts pooled"])

# Set TopK bag mean is very similar to the pooled mean.

,number_of_pairs,model_mean,human_mean
Bag,30,0.180333,0.516790
All contexts pooled,480,0.183958,0.504044


In [69]:
set_topk_bag_products = set_topk_bag[
    ["trigger", "query", "prediction", "human_rate"]
].copy()

set_topk_bag_products["bag_model_deviation"] = (
    set_topk_bag_products["prediction"] - set_topk_bag_model_mean
)

set_topk_bag_products["bag_human_deviation"] = (
    set_topk_bag_products["human_rate"] - set_topk_bag_human_mean
)

set_topk_bag_products["within_bag_product"] = (
    set_topk_bag_products["bag_model_deviation"]
    * set_topk_bag_products["bag_human_deviation"]
)

set_topk_bag_products["pooled_model_deviation"] = (
    set_topk_bag_products["prediction"] - set_topk_pooled_model_mean
)

set_topk_bag_products["pooled_human_deviation"] = (
    set_topk_bag_products["human_rate"] - set_topk_pooled_human_mean
)

set_topk_bag_products["pooled_product"] = (
    set_topk_bag_products["pooled_model_deviation"]
    * set_topk_bag_products["pooled_human_deviation"]
)

set_topk_bag_products

,trigger,query,prediction,human_rate,bag_model_deviation,bag_human_deviation,within_bag_product,pooled_model_deviation,pooled_human_deviation,pooled_product
960,cash,chalk,0.014,1.000000,-0.166333,0.483210,-0.080374,-0.169958,0.495956,-0.084292
961,cash,crayons,0.068,1.000000,-0.112333,0.483210,-0.054281,-0.115958,0.495956,-0.057510
962,cash,headphones,0.000,0.000000,-0.180333,-0.516790,0.093194,-0.183958,-0.504044,0.092723
963,cash,makeup,0.008,0.105263,-0.172333,-0.411527,0.070920,-0.175958,-0.398780,0.070169
964,cash,pen,0.990,1.000000,0.809667,0.483210,0.391239,0.806042,0.495956,0.399761
965,chalk,cash,0.002,0.000000,-0.178333,-0.516790,0.092161,-0.181958,-0.504044,0.091715
966,chalk,crayons,0.068,1.000000,-0.112333,0.483210,-0.054281,-0.115958,0.495956,-0.057510
967,chalk,headphones,0.000,0.200000,-0.180333,-0.316790,0.057128,-0.183958,-0.304044,0.055931
968,chalk,makeup,0.008,0.200000,-0.172333,-0.316790,0.054593,-0.175958,-0.304044,0.053499
969,chalk,pen,0.990,1.000000,0.809667,0.483210,0.391239,0.806042,0.495956,0.399761


In [70]:
set_topk_bag_numerator = math.fsum(
    set_topk_bag_products["within_bag_product"]
)

set_topk_bag_pooled_contribution = math.fsum(
    set_topk_bag_products["pooled_product"]
)

pd.DataFrame({
    "pairs_summed": [30, 30],
    "sum_of_products": [
        set_topk_bag_numerator,
        set_topk_bag_pooled_contribution
    ]
}, index=[
    "Within-bag numerator",
    "Bag contribution to pooled numerator"
])

,pairs_summed,sum_of_products
Within-bag numerator,30,2.550356
Bag contribution to pooled numerator,30,2.548970


In [71]:
set_topk_bag_products["bag_model_squared_deviation"] = (
    set_topk_bag_products["bag_model_deviation"] ** 2
)

set_topk_bag_products["bag_human_squared_deviation"] = (
    set_topk_bag_products["bag_human_deviation"] ** 2
)

set_topk_bag_products[
    [
        "trigger", "query",
        "bag_model_deviation", "bag_model_squared_deviation",
        "bag_human_deviation", "bag_human_squared_deviation"
    ]
]

,trigger,query,bag_model_deviation,bag_model_squared_deviation,bag_human_deviation,bag_human_squared_deviation
960,cash,chalk,-0.166333,0.027667,0.483210,0.233492
961,cash,crayons,-0.112333,0.012619,0.483210,0.233492
962,cash,headphones,-0.180333,0.032520,-0.516790,0.267072
963,cash,makeup,-0.172333,0.029699,-0.411527,0.169354
964,cash,pen,0.809667,0.655560,0.483210,0.233492
965,chalk,cash,-0.178333,0.031803,-0.516790,0.267072
966,chalk,crayons,-0.112333,0.012619,0.483210,0.233492
967,chalk,headphones,-0.180333,0.032520,-0.316790,0.100356
968,chalk,makeup,-0.172333,0.029699,-0.316790,0.100356
969,chalk,pen,0.809667,0.655560,0.483210,0.233492


In [72]:
set_topk_bag_model_squared_sum = math.fsum(
    set_topk_bag_products["bag_model_squared_deviation"]
)

set_topk_bag_human_squared_sum = math.fsum(
    set_topk_bag_products["bag_human_squared_deviation"]
)

set_topk_bag_denominator = math.sqrt(
    set_topk_bag_model_squared_sum
    * set_topk_bag_human_squared_sum
)

pd.DataFrame({
    "sum_squared_model_deviations": [
        set_topk_bag_model_squared_sum
    ],
    "sum_squared_human_deviations": [
        set_topk_bag_human_squared_sum
    ],
    "denominator": [set_topk_bag_denominator]
}, index=["Set Top-K within bag"])

,sum_squared_model_deviations,sum_squared_human_deviations,denominator
Set Top-K within bag,3.949337,5.737297,4.760096


In [73]:
set_topk_bag_pearson_manual = (
    set_topk_bag_numerator / set_topk_bag_denominator
)

print(pd.DataFrame({
    "numerator": [set_topk_bag_numerator],
    "denominator": [set_topk_bag_denominator],
    "pearson": [set_topk_bag_pearson_manual]
}, index=["Set Top-K within bag"]))

set_topk_bag_pearson_function = pearson(
    set_topk_bag["prediction"].tolist(),
    set_topk_bag["human_rate"].tolist()
)

print()
print("Existing function:", set_topk_bag_pearson_function)

print(
    "Results agree:",
    math.isclose(
        set_topk_bag_pearson_manual,
        set_topk_bag_pearson_function,
        rel_tol=1e-12
    )
)

                      numerator  denominator   pearson
Set Top-K within bag   2.550356     4.760096  0.535778

Existing function: 0.5357783039114599
Results agree: True


### Conclusions 

1. Set TopK has a slightly better Pearson within bag than No linking (0.536 versus 0.50).

2. Changing the reference means barely changes Set Top-K’s bag numerator.
    
    The sums were:
    - Using bag’s means: 2.550356
    - Using pooled means: 2.548970
    Set Top-K’s similar model means: 0.1803 within bag versus 0.1840 pooled.
    For No linking, those means differ more: 0.00323 within bag versus 0.01816 pooled. Its bag numerator consequently changes more in relative terms when using pooled means. 

## Centering and Standardizing within contexts 

Across all 480 Set Top-K observations, calculate the means and standard deviations separately within each context.

In [ ]:
# Center (means of zero)

columns = ["prediction", "human_rate"]

set_topk_context_means = (
    set_topk.groupby("context_id")[columns]
    .transform("mean")
)

set_topk_centered = set_topk[columns] - set_topk_context_means

set_topk_centered_pearson = pearson(
    set_topk_centered["prediction"].tolist(),
    set_topk_centered["human_rate"].tolist()
)

set_topk_centered.rename(columns={
    "prediction": "model_deviation",
    "human_rate": "human_deviation"
})

,model_deviation,human_deviation
960,-0.166333,0.483210
961,-0.112333,0.483210
962,-0.180333,-0.516790
963,-0.172333,-0.411527
964,0.809667,0.483210
...,...,...
1435,0.678667,0.540483
1436,-0.307333,0.040483
1437,-0.307333,-0.259517
1438,-0.287333,-0.459517


In [75]:
# Standardize (standard deviations of one)

set_topk_context_sds = (
    set_topk.groupby("context_id")[columns]
    .transform(lambda values: values.std(ddof=0))
)

set_topk_standardized = set_topk_centered / set_topk_context_sds

set_topk_standardized_pearson = pearson(
    set_topk_standardized["prediction"].tolist(),
    set_topk_standardized["human_rate"].tolist()
)

set_topk_standardized.rename(columns={
    "prediction": "model_z_score",
    "human_rate": "human_z_score"
})

,model_z_score,human_z_score
960,-0.458435,1.104951
961,-0.309604,1.104951
962,-0.497021,-1.181738
963,-0.474972,-0.941033
964,2.231541,1.104951
...,...,...
1435,1.588757,1.508526
1436,-0.719466,0.112990
1437,-0.719466,-0.724332
1438,-0.672646,-1.282546


In [76]:
# Compare

set_topk_context_correlations = []

for context, group in set_topk.groupby("context_id"):
    set_topk_context_correlations.append(
        pearson(
            group["prediction"].tolist(),
            group["human_rate"].tolist()
        )
    )

set_topk_mean_within = mean(set_topk_context_correlations)

pd.DataFrame({
    "pearson": [
        set_topk_pooled_manual,
        set_topk_centered_pearson,
        set_topk_standardized_pearson,
        set_topk_mean_within
    ]
}, index=[
    "Original pooled",
    "Context-centered pooled",
    "Context-standardized pooled",
    "Mean within-context"
])

,pearson
Original pooled,0.539816
Context-centered pooled,0.561473
Context-standardized pooled,0.585589
Mean within-context,0.585589


## Conclusions 

1. Removing context mean differences has the same effect for both Set TopK and No linking. 

    Centering increases Pearson by approximately:
    - No linking: +0.022
    - Set Top-K: +0.022

    Differences between context means alone does not distinguish the models' pooled versus within context Pearson gaps. 

2. Within context normalization affects the No linking Pearson correlation more than Set TopK

    After centering, standardization increases Pearson by:
    - No linking: +0.158
    - Set Top-K: +0.024

    The larger change for No linking occured when I removed differences in within-context scales.

3. Set Top-K’s advantage is larger in the pooled comparison.

    The difference between models is approximately:
    - Pooled Pearson: Set Top-K is 0.179 higher.
    - Mean within-context Pearson: Set Top-K is 0.044 higher.

# Log Likelihoods and Log Score 

Log score: Each trigger-query pair contributes equal weight. (This is also the case in Pearson and Spearman correlations). 
    
The analysis treats each trigger–query pair as an equally important prediction, even when pairs have different numbers of human responses.
    
For example:
- Pair A has 5 responses.
- Pair B has 20 responses.

With total response-level log-likelihood, Pair B contributes four times as many response terms. It therefore has more influence.

With the mean pair-level log score, first average within each pair, then average across pairs. A and B receive equal weight.


For one context–trigger–query pair:

- $n$: number of human responses.
- $k$: number of exclusion responses.
- $y = k/n$: observed human exclusion rate.
- $p$: model-predicted exclusion probability.

The total Bernoulli log-likelihood of the individual responses is:

$$
\ell = k\ln(p) + (n-k)\ln(1-p)
$$

The pair-level log score is the average log-likelihood per response:

$$
s = \frac{\ell}{n}
= \frac{k}{n}\ln(p) + \frac{n-k}{n}\ln(1-p)
= y\ln(p) + (1-y)\ln(1-p)
$$

$ y\ln(p) $ captures the exclusion 

$ (1-y)\ln(1-p) $ captures the non-exclusion

## Example: One context-trigger-query and one model. 

Bag context, cash → chalk, No linking model.

In [77]:
# Start with one context-trigger-query and one model. 
# Bag context, cash → chalk, No linking model.

log_score_example = results.loc[
    (results["model"] == "No linking structure")
    & (results["context_id"] == "bag")
    & (results["trigger"] == "cash")
    & (results["query"] == "chalk"),
    ["model", "context_id", "trigger", "query",
     "prediction", "human_rate"]
].copy()

log_score_example

,model,context_id,trigger,query,prediction,human_rate
0,No linking structure,bag,cash,chalk,0.000027,1.0


In [ ]:
# Unlike Pearson, log score does not subtract means. 

example_p = float(log_score_example["prediction"].iloc[0])

example_log_p = math.log(example_p)
example_log_not_p = math.log1p(-example_p)

pd.DataFrame({
    "model_probability": [
        example_p,
        1 - example_p
    ],
    "natural_log": [
        example_log_p,
        example_log_not_p
    ]
}, index=["Exclusion", "Non-exclusion"])

,model_probability,natural_log
Exclusion,0.000027,-10.517298
Non-exclusion,0.999973,-0.000027


In [79]:
example_y = float(log_score_example["human_rate"].iloc[0])

print("Model probability of exclusion:")
print(f"p = {example_p}")

print("\nModel probability of non-exclusion:")
print(f"1 - p = {1 - example_p}")

print("\nLog probabilities:")
print(f"ln(p) = {example_log_p}")
print(f"ln(1 - p) = {example_log_not_p}")

print("\nObserved human proportions:")
print(f"y = {example_y}")
print(f"1 - y = {1 - example_y}")

Model probability of exclusion:
p = 2.706422684422374e-05

Model probability of non-exclusion:
1 - p = 0.9999729357731558

Log probabilities:
ln(p) = -10.517297744750977
ln(1 - p) = -2.7064593087019146e-05

Observed human proportions:
y = 1.0
1 - y = 0.0


In [ ]:
# Multiply the log probabilities by the observed human proportions.

example_exclusion_term = example_y * example_log_p

example_non_exclusion_term = (
    (1 - example_y) * example_log_not_p
)

print("Exclusion term:")
print(
    f"y * ln(p) = {example_y} * {example_log_p}"
    f" = {example_exclusion_term}"
)

print()
print("Non-exclusion term:")
print(
    f"(1 - y) * ln(1 - p) = {1 - example_y} * {example_log_not_p}"
    f" = {example_non_exclusion_term}"
)

Exclusion term:
y * ln(p) = 1.0 * -10.517297744750977 = -10.517297744750977

Non-exclusion term:
(1 - y) * ln(1 - p) = 0.0 * -2.7064593087019146e-05 = -0.0


In [82]:
# Add the two terms to get the log score.

example_log_score = (
    example_exclusion_term + example_non_exclusion_term
)

print(
    f"Log score = {example_exclusion_term}"
    f" + {example_non_exclusion_term}"
)

print(f"Log score = {example_log_score}")

Log score = -10.517297744750977 + -0.0
Log score = -10.517297744750977


In [85]:
saved_log_score = results.loc[
    log_score_example.index, "item_log_score"
].iloc[0]

print("Our calculation:", example_log_score)
print("Saved log score:", saved_log_score)
print(
    "Results agree:",
    math.isclose(example_log_score, saved_log_score, rel_tol=1e-12)
)

print()
print("The strongly negative score reflects that humans always excluded, " \
"while the model assigned exclusion an extremely small probability.")

Our calculation: -10.517297744750977
Saved log score: -10.517297744750977
Results agree: True

The strongly negative score reflects that humans always excluded, while the model assigned exclusion an extremely small probability.


## Log score across all observations (No linking model)

In [94]:
from build_results_viewer import EPSILON

log_score_terms = no_linking[
    ["context_id", "trigger", "query",
     "prediction", "human_rate", "item_log_score"]
].copy()

# Match the existing function's protection against log(0).
log_score_terms["p_used"] = log_score_terms["prediction"].clip(
    EPSILON, 1 - EPSILON
)

# log_score_terms["ln_p"] = log_score_terms["p_used"].map(math.log)

# log_score_terms["ln_one_minus_p"] = log_score_terms["p_used"].map(
#     lambda p: math.log1p(-p)
# )

log_score_terms["exclusion_term"] = (
    log_score_terms["human_rate"] 
    * log_score_terms["p_used"].map(math.log)
)

log_score_terms["non_exclusion_term"] = (
    (1 - log_score_terms["human_rate"])
    * log_score_terms["p_used"].map(lambda p: math.log1p(-p))
)

log_score_terms["calculated_log_score"] = (
    log_score_terms["exclusion_term"]
    + log_score_terms["non_exclusion_term"]
)

print("Number of pairs:", len(log_score_terms))
print("Each row shows the log score calculation for one context-trigger-query pair.")
log_score_terms

Number of pairs: 480
Each row shows the log score calculation for one context-trigger-query pair.


,context_id,trigger,query,prediction,human_rate,item_log_score,p_used,exclusion_term,non_exclusion_term,calculated_log_score
0,bag,cash,chalk,2.706423e-05,1.000000,-10.517298,2.706423e-05,-10.517298,-0.000000e+00,-10.517298
1,bag,cash,crayons,1.865203e-04,1.000000,-8.586970,1.865203e-04,-8.586970,-0.000000e+00,-8.586970
2,bag,cash,headphones,2.221567e-06,0.000000,-0.000002,2.221567e-06,-0.000000,-2.221569e-06,-0.000002
3,bag,cash,makeup,1.860095e-05,0.105263,-1.146574,1.860095e-05,-1.146558,-1.664311e-05,-1.146574
4,bag,cash,pen,1.916255e-02,1.000000,-3.954797,1.916255e-02,-3.954797,-0.000000e+00,-3.954797
...,...,...,...,...,...,...,...,...,...,...
475,transport,roller blades,car,1.242291e-01,1.000000,-2.085628,1.242291e-01,-2.085628,-0.000000e+00,-2.085628
476,transport,roller blades,chairs,5.349480e-07,0.500000,-7.220548,5.349480e-07,-7.220548,-2.674741e-07,-7.220548
477,transport,roller blades,hammer,1.395912e-05,0.200000,-2.235887,1.395912e-05,-2.235876,-1.116737e-05,-2.235887
478,transport,roller blades,lawnmower,3.070479e-04,0.000000,-0.000307,3.070479e-04,-0.000000,-3.070951e-04,-0.000307


In [96]:
# Now average the log scores across all 480 context-trigger-query pairs to get the mean log score for the No linking model.
# Every pair contributes one score with equal weight, regardless of how many human responses produced its exclusion rate.


no_linking_sum_pair_scores = math.fsum(
    log_score_terms["calculated_log_score"]
)

no_linking_mean_log_score = mean(
    log_score_terms["calculated_log_score"].tolist()
)

print(f"Sum of pair scores is the sum of item_log_score.")
print(f"Mean log score is the average of the pair scores: {no_linking_sum_pair_scores} / {len(log_score_terms)} = {no_linking_mean_log_score}")

pd.DataFrame({
    "number_of_pairs": [len(log_score_terms)],
    "sum_of_pair_scores": [no_linking_sum_pair_scores],
    "mean_log_score": [no_linking_mean_log_score]
}, index=["No linking"])

Sum of pair scores is the sum of item_log_score.
Mean log score is the average of the pair scores: -1544.387761643921 / 480 = -3.2174745034248358


,number_of_pairs,sum_of_pair_scores,mean_log_score
No linking,480,-1544.387762,-3.217475


### Compare to predicting 0.5 exclusion probability for every pair.

In [97]:
constant_half_score = math.log(0.5)

pd.DataFrame({
    "mean_log_score": [
        no_linking_mean_log_score,
        constant_half_score
    ]
}, index=[
    "No linking",
    "Constant p = 0.5"
])

,mean_log_score
No linking,-3.217475
Constant p = 0.5,-0.693147


No linking scores worse than assigning 50% exclusion probability to every pair.

No linking has a positive pooled Pearson correlation with human exclusion rates, but its predicted probability magnitudes receive a much worse mean log score than a constant 0.5 predictor.

Argument: A model can track which pairs have higher human rates while assigning poor probabilities to those responses.

Next, inspect which pairs contribute the largest penalties and see why.

## Worst log score, No linking

In [98]:
worst_no_linking = (
    log_score_terms
    .sort_values("calculated_log_score")
    .head(10)
)

worst_no_linking[
    [
        "context_id", "trigger", "query",
        "prediction", "human_rate",
        "exclusion_term", "non_exclusion_term",
        "calculated_log_score"
    ]
]

,context_id,trigger,query,prediction,human_rate,exclusion_term,non_exclusion_term,calculated_log_score
259,mall,gift shop,pretzel stand,0.000005,1.000000,-12.289368,-0.000000e+00,-12.289368
249,mall,coffee shop,pretzel stand,0.000005,1.000000,-12.289368,-0.000000e+00,-12.289368
0,bag,cash,chalk,0.000027,1.000000,-10.517298,-0.000000e+00,-10.517298
21,bag,makeup,chalk,0.000027,1.000000,-10.517298,-0.000000e+00,-10.517298
244,mall,burger joint,pretzel stand,0.000005,0.842105,-10.348941,-7.263803e-07,-10.348942
207,gym,weightlifting,meditation,0.000035,1.000000,-10.263705,-0.000000e+00,-10.263705
219,hot,couch,shower,0.000047,1.000000,-9.975464,-0.000000e+00,-9.975464
100,cold,heater,blankets,0.000007,0.833333,-9.878743,-1.184431e-06,-9.878745
264,mall,grocery store,pretzel stand,0.000005,0.800000,-9.831494,-9.200818e-07,-9.831495
446,throw,tires,boomerang,0.000062,1.000000,-9.687407,-0.000000e+00,-9.687407


Looking at mall, gift shop -> pretzel stand:
-  Model exclusion probability $p$ is 0.000005.
- Human exclusion rate $y$ is 1.0. Every observed response was an exclusion.

The model assigned almost no probability to the outcome every human produced, so it receives a strongly negative score.

For comparison, predicting $p=0.5$ would give this pair a score of −0.693147, which is much higher than the No linking score.

The broader pattern: Very small model exclusion probabilities paired with high human exclusion rates produce large penalties.

In [101]:
# Calculate how much these 10 lowest-scoring pairs contribute to the overall log score mean.

remaining_pairs = log_score_terms.drop(
    index=worst_no_linking.index
)

worst_sum = math.fsum(worst_no_linking["calculated_log_score"])
remaining_sum = math.fsum(remaining_pairs["calculated_log_score"])
total_sum = math.fsum(log_score_terms["calculated_log_score"])

total_pairs = len(log_score_terms)

contribution_table = pd.DataFrame({
    "number_of_pairs": [
        len(worst_no_linking),
        len(remaining_pairs),
        total_pairs
    ],
    "sum_of_scores": [
        worst_sum,
        remaining_sum,
        total_sum
    ],
    "contribution_to_overall_mean": [
        worst_sum / total_pairs,
        remaining_sum / total_pairs,
        total_sum / total_pairs
    ],
    "share_of_total_penalty_percent": [
        100 * worst_sum / total_sum,
        100 * remaining_sum / total_sum,
        100
    ]
}, index=[
    "10 lowest-scoring pairs",
    "Remaining pairs",
    "All pairs"
])

print("sum_of_scores: adds the scores within each group.")
print("contribution_to_overall_mean: divides the sum of scores by the total number of pairs (480).")
print("share_of_total_penalty_percent: divides the sum of scores by the total sum of scores (all pairs).  [ 100 * (-105.6 / -1544.4) ] = 6.8% ]")
contribution_table

sum_of_scores: adds the scores within each group.
contribution_to_overall_mean: divides the sum of scores by the total number of pairs (480).
share_of_total_penalty_percent: divides the sum of scores by the total sum of scores (all pairs).  [ 100 * (-105.6 / -1544.4) ] = 6.8% ]


,number_of_pairs,sum_of_scores,contribution_to_overall_mean,share_of_total_penalty_percent
10 lowest-scoring pairs,10,-105.599089,-0.219998,6.837602
Remaining pairs,470,-1438.788673,-2.997476,93.162398
All pairs,480,-1544.387762,-3.217475,100.000000


### Conclusions 

1. The largest penalties occur when humans frequently exclude but No linking assigns exclusion almost no probability.

    Across these 10 pairs:
    - Human exclusion rates range from 0.8 to 1.0.
    - Model exclusion probabilities are approximately 0.000005–0.000062.
    - Log scores range from approximately −12.29 to −9.69.

    Clear examples of severe underprediction of exclusion.

2. The exclusion term produces almost all of the penalty. 

    The large negative values come from: $ y ln(p) $. $y$ is high and $p$ is very small. 

3. These 10 worst log score pairs do not account for most of the overall log score. 

    Their summed score is approximately −105.6, compared with −1544.4 across all pairs. 
    
    They represent:
    - 2.08% of pairs
    - Approximately 6.84% of the total negative-score magnitude

    The remaining 470 pairs still account for 93.16% of that magnitude. No linking’s poor overall score therefore cannot be attributed solely to these 10 extreme cases.

Note: Within a context, No linking gives a query the same prediction across triggers! 


## Highest log scores, No linking

In [102]:
best_no_linking = (
    log_score_terms
    .sort_values("calculated_log_score", ascending=False)
    .head(10)
)

best_no_linking[
    [
        "context_id", "trigger", "query",
        "prediction", "human_rate",
        "exclusion_term", "non_exclusion_term",
        "calculated_log_score"
    ]
]

,context_id,trigger,query,prediction,human_rate,exclusion_term,non_exclusion_term,calculated_log_score
267,mall,pretzel stand,electronics store,1.058005e-07,0.0,-0.0,-1.058005e-07,-1.058005e-07
262,mall,grocery store,electronics store,1.058005e-07,0.0,-0.0,-1.058005e-07,-1.058005e-07
241,mall,burger joint,electronics store,1.058005e-07,0.0,-0.0,-1.058005e-07,-1.058005e-07
450,transport,car,chairs,5.349480e-07,0.0,-0.0,-5.349481e-07,-5.349481e-07
471,transport,motorcycle,chairs,5.349480e-07,0.0,-0.0,-5.349481e-07,-5.349481e-07
151,fridge,juice,meat,1.179044e-06,0.0,-0.0,-1.179045e-06,-1.179045e-06
167,fridge,milk,meat,1.179044e-06,0.0,-0.0,-1.179045e-06,-1.179045e-06
172,fridge,water,meat,1.179044e-06,0.0,-0.0,-1.179045e-06,-1.179045e-06
156,fridge,ketchup,meat,1.179044e-06,0.0,-0.0,-1.179045e-06,-1.179045e-06
10,bag,crayons,cash,1.226866e-06,0.0,-0.0,-1.226867e-06,-1.226867e-06


# Jason Notes

- Pearson says: When this colum item is X standard deviations above the mean the other column item is also X standard deviations above its mean.
- Is the mean human rating meaningful? Across all contexts, no! Within context, yes! 
- Look at the human rate. The leevel of uncertainly of the trigger-query exclusion depends on how many people answered that trial. 
- Reminder for Jason: gitter by binomial variance 
- For each context:
    - Number of participants
    - Do the number of participants that saw an item pattern the same way?
- Three ways to do averaging:
    - Arithemtic
    - Geometric 
    - Harmonic 
- Look at the distribution of the 30 trigger-query pairs in a context for the human rate and No linking model. 
    - Is deviation from the mean a thing to care about?
- Leave the log score as -1500. Don't divide by 480. 
- Do log score on a model that predicts exclusion with 0, 0.1, 0.2, ..., 1. 
    - The maximum likelihood estimate should be the global human mean. 
- Log likelihood is a comparison across models 
- Correlation is a measure of understanding within a model. 